# Configuración para Google Colab

Ejecuta esta celda para instalar las dependencias necesarias en el entorno de Colab.

In [ ]:
!pip install mediapipe opencv-python scikit-learn tqdm matplotlib pandas numpy torch

### Instrucciones de carga de datos:
1. Sube tu carpeta `src/`, `data/` y `output/` a Colab.
2. Si usas un ZIP, súbelo y descomprímelo con: `!unzip project.zip`.

# Entrenamiento de Modelo ST-GCN (Versión Colab/CUDA)

Este notebook realiza el entrenamiento del modelo **Spatio-Temporal Graph Convolutional Network (ST-GCN)**.

In [ ]:
import os
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from datetime import datetime

# Asegurar que el directorio raiz esta en el path para importar src
ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.stgcn.stgcn_model import RealSTGCN
from src.stgcn.hand_graph import build_adjacency_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

In [ ]:
CONFIG = {
    "MANIFEST": "output/train_manifest_stgcn_secuencias.csv",
    "SECUENCIAS_DIR": "data/processed/secuencias_stgcn",
    "MODEL_OUTPUT": "output/models/stgcn_gesture_model_best.pth",
    "BATCH_SIZE": 64, # Aumentado para GPU
    "EPOCHS": 30,
    "LR": 0.001,
    "SEED": 42
}

Path("output/models").mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["SEED"])

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

if not Path(CONFIG["MANIFEST"]).exists():
    raise FileNotFoundError(f"No se encuentra el manifiesto en {CONFIG['MANIFEST']}. Asegúrate de subir la carpeta output/.")

df = pd.read_csv(CONFIG["MANIFEST"])
df = df[df['quality_flag'] != 'excluded'].copy()
df['label'] = df['label'].fillna('unknown').astype(str)

le = LabelEncoder()
df['label_idx'] = le.fit_transform(df['label'])
unique_labels = le.classes_.tolist()
label_to_idx = {name: i for i, name in enumerate(unique_labels)}

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(df['label_idx']),
    y=df['label_idx']
)
weights_tensor = torch.FloatTensor(class_weights).to(device)

train_df = df.sample(frac=0.8, random_state=CONFIG["SEED"])
val_df = df.drop(train_df.index)
print(f"Dataset listo: {len(df)} muestras, {len(unique_labels)} clases.")

In [ ]:
adj = build_adjacency_matrix().to(device)
model = RealSTGCN(
    num_classes=len(unique_labels),
    adjacency=adj,
    in_channels=3,
    dropout=0.3
).to(device)
print("Modelo ST-GCN listo en GPU.")

In [ ]:
class STGCNDataset(Dataset):
    def __init__(self, manifest_df, base_dir, label_map, augment=False):
        self.df = manifest_df
        self.base_dir = Path(base_dir)
        self.label_map = label_map
        self.augment = augment
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = Path(row['path_secuencia'])
        if not path.exists(): path = self.base_dir / path.name
        seq = np.load(path).astype(np.float32)
        if self.augment:
            num_nodes = seq.shape[1]
            mask = np.random.rand(num_nodes) > 0.1
            seq[:, ~mask, :] = 0.0
        x = torch.from_numpy(np.transpose(seq, (2, 0, 1))).float()
        y = torch.tensor(self.label_map[str(row['label'])], dtype=torch.long)
        return x, y

train_ds = STGCNDataset(train_df, CONFIG["SECUENCIAS_DIR"], label_to_idx, augment=True)
val_ds = STGCNDataset(val_df, CONFIG["SECUENCIAS_DIR"], label_to_idx, augment=False)
train_loader = DataLoader(train_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=2)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["LR"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['EPOCHS'])

print("Iniciando entrenamiento...")
best_acc = 0.0
for epoch in range(CONFIG['EPOCHS']):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs, _ = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward(); optimizer.step()
        total_loss += loss.item()
    
    scheduler.step()
    
    model.eval()
    correct = 0; total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs, _ = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0); correct += (predicted == labels).sum().item()
    
    val_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{CONFIG['EPOCHS']}] Loss: {total_loss/len(train_loader):.4f} Val Acc: {val_acc:.2f}%")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), CONFIG["MODEL_OUTPUT"])
        print(f"  --> Guardado mejor modelo ({best_acc:.2f}%)")